In [9]:
import pandas as pd
import featuretools as ft
from featuretools.primitives import Day, Weekday, IsWeekend, Hour, Mean, Sum, Count, Max, Std, Skew
import json
import numpy as np

In [10]:
users_df = pd.read_csv('../Final/users.csv')
accounts_df = pd.read_csv('../Final/accounts.csv')
transactions_df = pd.read_csv('../Final/transactions.csv')
monthly_stats_df = pd.read_csv('../analytics/user_monthly_stats.csv')
all_time_stats_df = pd.read_csv('../analytics/user_all_time_stats.csv')

def expand_json_categories(df, column_name):
    json_as_df = df[column_name].apply(lambda x: json.loads(x.replace("'", '"')) if isinstance(x, str) else x).apply(pd.Series)
    json_as_df = json_as_df.fillna(0)
    return pd.concat([df.drop(columns=[column_name]), json_as_df], axis=1)

transactions_df['date'] = pd.to_datetime(transactions_df['date'])
monthly_stats_df['month_start_date'] = pd.to_datetime(monthly_stats_df['month_start_date'])
monthly_stats_df = expand_json_categories(monthly_stats_df, 'spending_by_category')

es = ft.EntitySet(id="financial__system")

In [11]:
es.add_dataframe(dataframe_name="users", dataframe=users_df, index="user_id")

es.add_dataframe(dataframe_name="accounts", dataframe=accounts_df, index="account_id")

es.add_dataframe(dataframe_name="transactions", dataframe=transactions_df, 
                 index="transaction_id", time_index="date")

es.add_dataframe(dataframe_name="monthly_stats", dataframe=monthly_stats_df, 
                 index="stats_id", time_index="month_start_date")

es.add_dataframe(dataframe_name="all_time_stats", dataframe=all_time_stats_df, index="stats_id")

es.add_relationship("accounts", "account_id", "users", "account_id")
es.add_relationship("accounts", "account_id", "transactions", "account_id")
es.add_relationship("accounts", "account_id", "monthly_stats", "account_id")
es.add_relationship("accounts", "account_id", "all_time_stats", "account_id")

/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33

Entityset: financial__system
  DataFrames:
    users [Rows: 388, Columns: 9]
    accounts [Rows: 388, Columns: 7]
    transactions [Rows: 22602, Columns: 10]
    monthly_stats [Rows: 5880, Columns: 28]
    all_time_stats [Rows: 306, Columns: 6]
  Relationships:
    users.account_id -> accounts.account_id
    transactions.account_id -> accounts.account_id
    monthly_stats.account_id -> accounts.account_id
    all_time_stats.account_id -> accounts.account_id

# Time-Based Amount Features

In [12]:
trans_primitives = [Day, Weekday, IsWeekend]
agg_primitives = [Mean, Sum, Count, Max, Std]

feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="users",
    trans_primitives=trans_primitives,
    agg_primitives=agg_primitives,
    max_depth=2,
    verbose=True
)

numeric_cols = feature_matrix.select_dtypes(include=[np.number]).columns
feature_matrix[numeric_cols] = feature_matrix[numeric_cols].fillna(0)

/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/featuretools/synthesis/dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  trans_primitives: ['day', 'is_weekend', 'weekday']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)


Built 129 features
Elapsed: 00:00 | Progress: 100%|██████████


/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/featuretools/computational_backends/feature_set_calculator.py:781: FutureWarning: The provided callable <function max at 0x7598c08bf280> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  to_merge = base_frame.groupby(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/featuretools/computational_backends/feature_set_calculator.py:781: FutureWarning: The provided callable <function mean at 0x7598c08bfb80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  to_merge = base_frame.groupby(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/featuretools/computational_backends/feature_set_calculator.py:781: FutureWarning: The provided callable <function

#### Remove Redundant Features - Feature Selection

SUM and MEAN are simmilar so that it can make noise if account has only one transaction.

Steps are:
1. Create a correlation matrix.
1. Select the upper triangle of the correlation matric
1. Identify columns with correlation higher than 0.95

In [13]:
corr_matrix = feature_matrix.corr(numeric_only=True).abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

print(f"Dropping {len(to_drop)} highly correlated features.")
feature_matrix_reduced = feature_matrix.drop(columns=to_drop)

Dropping 62 highly correlated features.


#### Encode Categorical Data

Encode categorical features

1. Identify which features actually remain in your reduced matrix
1. Encode using ONLY the definitions that still exist
1. Limits one-hot encoding to top 10 categories to prevent "column explosion"

In [15]:
remaining_features = [
    f for f in feature_defs 
    if any(name in feature_matrix_reduced.columns for name in f.get_feature_names())
]

feature_matrix_reduced.ww.init()

fm_encoded, features_defs_encoded = ft.encode_features(
    feature_matrix_reduced, 
    remaining_features,
    top_n=10 
)

print(f"Final feature count after encoding: {fm_encoded.shape[1]}")

Final feature count after encoding: 106


/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/workspaces/python/env/featuretools-env/lib/python3.9/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
